In [ ]:
from google.colab import files
uploaded = files.upload()

Saving uni-20260224T175509Z-1-001.zip to uni-20260224T175509Z-1-001.zip


In [ ]:
!unzip uni-20260224T175509Z-1-001.zip

Archive:  uni-20260224T175509Z-1-001.zip
  inflating: uni/UNIPROT66.txt       
  inflating: uni/uniprot45.txt       
  inflating: uni/uniprot49.txt       
  inflating: uni/uniprot41.txt       
  inflating: uni/uniprot37.txt       
  inflating: uni/uniprot53.txt       
  inflating: uni/uniprot56.txt       
  inflating: uni/uniprot44.txt       
  inflating: uni/uniprot64.txt       
  inflating: uni/uniprot42.txt       
  inflating: uni/uniprot50.txt       
  inflating: uni/uniprot39.txt       
  inflating: uni/uniprot58.txt       
  inflating: uni/uniprot60.txt       
  inflating: uni/uniprot61.txt       
  inflating: uni/uniprot.txt         
  inflating: uni/uniprot52.txt       
  inflating: uni/uniprot65.txt       
  inflating: uni/uniprot62.txt       
  inflating: uni/uniprot48.txt       
  inflating: uni/uniprot38.txt       
  inflating: uni/uniprot59.txt       
  inflating: uni/uniprot40.txt       
  inflating: uni/uniprot51.txt       
  inflating: uni/uniprot57.txt       
  inflati

In [ ]:
!cat uni/*.txt > combined_proteins.fasta

In [ ]:
FASTA_PATH = "combined_proteins.fasta"

In [ ]:
!head -3 combined_proteins.fasta
!grep -c "^>" combined_proteins.fasta

>sp|P39059|COFA1_HUMAN Collagen alpha-1(XV) chain OS=Homo sapiens OX=9606 GN=COL15A1 PE=1 SV=2
MAPRRNNGQCWCLLMLLSVSTPLPAVTQTRGATETASQGHLDLTQLIGVPLPSSVSFVTG
YGGFPAYSFGPGANVGRPARTLIPSTFFRDFAISVVVKPSSTRGGVLFAITDAFQKVIYL
68


In [ ]:
!pip -q install biopython pandas

import pandas as pd
from Bio import SeqIO

def cleavage_sites_trypsin(seq: str, proline_exception: bool = True):
    cuts = [0]
    for i, aa in enumerate(seq[:-1]):
        if aa in ("K", "R"):
            if proline_exception and seq[i+1] == "P":
                continue
            cuts.append(i+1)
    cuts.append(len(seq))
    return sorted(set(cuts))

def digest_protein(seq: str, missed_cleavages=2, min_len=7, max_len=35, proline_exception=True):
    seq = str(seq).replace("*", "").upper()
    cuts = cleavage_sites_trypsin(seq, proline_exception=proline_exception)
    peptides = []
    for start_i in range(len(cuts)-1):
        for mc in range(missed_cleavages + 1):
            end_i = start_i + 1 + mc
            if end_i >= len(cuts):
                continue
            start = cuts[start_i]
            end = cuts[end_i]
            pep = seq[start:end]
            if min_len <= len(pep) <= max_len:
                peptides.append(pep)
    return peptides

FASTA_PATH = "combined_proteins.fasta"

records = list(SeqIO.parse(FASTA_PATH, "fasta"))
print("Loaded proteins:", len(records))

rows = []
for rec in records:
    for pep in digest_protein(rec.seq, missed_cleavages=2, min_len=7, max_len=35):
        rows.append({"protein_id": rec.id, "peptide": pep, "pep_len": len(pep)})

df = pd.DataFrame(rows).drop_duplicates()
print("Total peptides:", len(df))

df.to_csv("step1_digestion_peptides.csv", index=False)
print("Saved: step1_digestion_peptides.csv")

Loaded proteins: 68
Total peptides: 24804
Saved: step1_digestion_peptides.csv


In [ ]:
import re
import pandas as pd
from Bio import SeqIO
from collections import defaultdict

FASTA_PATH = "combined_proteins.fasta"

def parse_uniprot_species(desc: str):

    os_match = re.search(r'OS=([^=]+?)\sOX=', desc)
    ox_match = re.search(r'OX=(\d+)', desc)
    species = os_match.group(1).strip() if os_match else "UNKNOWN"
    taxid = ox_match.group(1) if ox_match else "NA"
    return species, taxid


prot2species = {}
for rec in SeqIO.parse(FASTA_PATH, "fasta"):
    species, taxid = parse_uniprot_species(rec.description)
    prot2species[rec.id] = (species, taxid)

print("Example mapping:", list(prot2species.items())[:3])


df = pd.read_csv("step1_digestion_peptides.csv")


df["species"] = df["protein_id"].map(lambda x: prot2species.get(x, ("UNKNOWN","NA"))[0])
df["taxid"]   = df["protein_id"].map(lambda x: prot2species.get(x, ("UNKNOWN","NA"))[1])


pep2species = df.groupby("peptide")["species"].agg(lambda s: sorted(set(s))).reset_index()
pep2species["n_species"] = pep2species["species"].apply(len)


pep2species.to_csv("step2_peptide_species_map.csv", index=False)
df.to_csv("step2_peptides_with_species.csv", index=False)

print("Unique peptides:", pep2species.shape[0])
print("Peptides found in exactly 1 species:", (pep2species["n_species"]==1).sum())
print("Saved: step2_peptide_species_map.csv, step2_peptides_with_species.csv")

Example mapping: [('sp|P39059|COFA1_HUMAN', ('Homo sapiens', '9606')), ('sp|Q8NFW1|COMA1_HUMAN', ('Homo sapiens', '9606')), ('sp|Q9UMD9|COHA1_HUMAN', ('Homo sapiens', '9606'))]
Unique peptides: 13840
Peptides found in exactly 1 species: 11762
Saved: step2_peptide_species_map.csv, step2_peptides_with_species.csv


In [ ]:
import pandas as pd

pep_map = pd.read_csv("step2_peptide_species_map.csv")
pep_occ = pd.read_csv("step2_peptides_with_species.csv")

unique_peps = pep_map[pep_map["n_species"] == 1].copy()


def first_species(x):

    s = str(x)

    s = s.strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1].strip()

        s = s.strip("'").strip('"')

    return s.split(",")[0].strip().strip("'").strip('"')

unique_peps["unique_species"] = unique_peps["species"].apply(first_species)

print("Species-unique peptides:", len(unique_peps))
unique_peps.head()

Species-unique peptides: 11762


,peptide,species,n_species,unique_species
0,AAARRLAYHGGNTNTGDALR,['Homo sapiens'],1,Homo sapiens
1,AAASGSR,['Mus musculus'],1,Mus musculus
2,AAASGSRGPGELGAPGPGTVALAEQCAR,['Mus musculus'],1,Mus musculus
3,AACGKVR,['Homo sapiens'],1,Homo sapiens
4,AACGKVRGSENCALGGQCVK,['Homo sapiens'],1,Homo sapiens


In [ ]:
import re
import pandas as pd
from Bio import SeqIO

FASTA_PATH = "combined_proteins.fasta"

def extract_pe(description):
    match = re.search(r"\bPE=(\d)\b", description)
    return int(match.group(1)) if match else None

protein_pe = {}
protein_existence_score = {}

for record in SeqIO.parse(FASTA_PATH, "fasta"):
    pe = extract_pe(record.description)
    protein_pe[record.id] = pe
    protein_existence_score[record.id] = (6 - pe) if pe else None
    # PE=1 → 5 (best), PE=5 → 1 (worst)

print("Parsed PE for", len(protein_pe), "proteins")

pep_occ = pd.read_csv("step2_peptides_with_species.csv")


def clean_species(x):
    x = str(x)
    if x.startswith("[") and x.endswith("]"):
        x = x[1:-1]
    return x.strip().strip("'").strip('"')

pep_occ["species"] = pep_occ["species"].apply(clean_species)

pep_occ["protein_existence_score"] = pep_occ["protein_id"].map(protein_existence_score)

pep_map = pd.read_csv("step2_peptide_species_map.csv")

unique_peptides = pep_map[pep_map["n_species"] == 1].copy()

unique_set = set(unique_peptides["peptide"])

unique_occ = pep_occ[pep_occ["peptide"].isin(unique_set)].copy()


peptide_scores = (
    unique_occ.groupby(["peptide", "species"])
    .agg(
        peptide_score=("protein_existence_score", "max"),
        n_proteins=("protein_id", "nunique")
    )
    .reset_index()
)

peptide_scores.to_csv("step3_unique_peptides_scored.csv", index=False)

print("Saved: step3_unique_peptides_scored.csv")


species_summary = (
    peptide_scores.groupby("species")
    .agg(
        unique_peptide_count=("peptide", "nunique"),
        sum_peptide_score=("peptide_score", "sum"),
        mean_peptide_score=("peptide_score", "mean")
    )
    .reset_index()
    .sort_values("unique_peptide_count", ascending=False)
)

species_summary.to_csv("step3_species_summary.csv", index=False)

print("Saved: step3_species_summary.csv")



protein_table = pd.DataFrame({
    "protein_id": list(protein_pe.keys()),
    "PE": [protein_pe[p] for p in protein_pe],
    "protein_existence_score": [protein_existence_score[p] for p in protein_pe]
})

protein_table.to_csv("step3_protein_existence_scores.csv", index=False)

print("Saved: step3_protein_existence_scores.csv")

print("\nSpecies Summary Preview:")
print(species_summary.head())

Parsed PE for 67 proteins
Saved: step3_unique_peptides_scored.csv
Saved: step3_species_summary.csv
Saved: step3_protein_existence_scores.csv

Species Summary Preview:
                   species  unique_peptide_count  sum_peptide_score  \
10            Mus musculus                  4322              20367   
8             Homo sapiens                  2804              14020   
16              Sus scrofa                  1329               2658   
3   Canis lupus familiaris                   548               1660   
9          Jaculus jaculus                   523               1046   

    mean_peptide_score  
10            4.712402  
8             5.000000  
16            2.000000  
3             3.029197  
9             2.000000  


In [ ]:
import pandas as pd


pep_map = pd.read_csv("step2_peptide_species_map.csv")
pep_occ = pd.read_csv("step2_peptides_with_species.csv")


unique_peptides = pep_map[pep_map["n_species"] == 1][["peptide"]]

unique_species_map = (
    unique_peptides
    .merge(pep_occ[["peptide","species","taxid"]].drop_duplicates(),
           on="peptide")
)

unique_counts = (
    unique_species_map
    .groupby(["species","taxid"])["peptide"]
    .nunique()
    .reset_index(name="number_unique_peptides")
    .sort_values("number_unique_peptides", ascending=False)
)

unique_counts

,species,taxid,number_unique_peptides
10,Mus musculus,10090,4322
8,Homo sapiens,9606,2804
16,Sus scrofa,9823,1329
3,Canis lupus familiaris,9615,548
9,Jaculus jaculus,51337,523
2,Camelus dromedarius,9838,499
12,Ovis aries,9940,317
11,Oryctolagus cuniculus,9986,316
1,Bos taurus,9913,252
7,Gallus gallus,9031,223


In [ ]:
import pandas as pd

pep_occ = pd.read_csv("step2_peptides_with_species.csv")

TOTAL_SPECIES = 16

protein_species = (
    pep_occ[["protein_id","species"]]
    .drop_duplicates()
    .groupby("protein_id")["species"]
    .nunique()
    .reset_index(name="n_species_protein")
)

protein_species["protein_existence_score"] = (
    protein_species["n_species_protein"] / TOTAL_SPECIES
)

protein_species.head()

,protein_id,n_species_protein,protein_existence_score
0,sp|A2AX52|CO6A4_MOUSE,1,0.0625
1,sp|A6H584|CO6A5_MOUSE,1,0.0625
2,sp|A6QPB3|COHA1_BOVIN,1,0.0625
3,sp|A8TX70|CO6A5_HUMAN,1,0.0625
4,sp|O35206|COFA1_MOUSE,1,0.0625


In [ ]:
pep_map = pd.read_csv("step2_peptide_species_map.csv")

pep_map["peptide_existence_score"] = (
    pep_map["n_species"] / TOTAL_SPECIES
)

pep_map[["peptide","n_species","peptide_existence_score"]].head()

,peptide,n_species,peptide_existence_score
0,AAARRLAYHGGNTNTGDALR,1,0.0625
1,AAASGSR,1,0.0625
2,AAASGSRGPGELGAPGPGTVALAEQCAR,1,0.0625
3,AACGKVR,1,0.0625
4,AACGKVRGSENCALGGQCVK,1,0.0625


In [ ]:
from google.colab import drive
drive.mount('/content/drive')




Mounted at /content/drive


In [ ]:
!cp step2_peptides_with_species.csv /content/drive/MyDrive/
!cp step2_peptide_species_map.csv /content/drive/MyDrive/